## GPT Architecture

This project is "trying" to replicate the GPT2 model, oit has the following configuration:

GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size             (Tokens)
    "context_length": 1024, # Context length              (Num of words process at the time)
    "emb_dim": 768,         # Embedding dimension         (Embedding)  
    "n_heads": 12,          # Number of attention heads   (Attention)
    "n_layers": 12,         # Number of layers            (MLP)
    "drop_rate": 0.1,       # Dropout rate                (Regularization)
    "qkv_bias": False       # Query-Key-Value bias        (No bias)
}

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size             (Tokens)
    "context_length": 1024, # Context length              (Num of words process at the time)
    "emb_dim": 768,         # Embedding dimension         (Embedding)  
    "n_heads": 12,          # Number of attention heads   (Attention)
    "n_layers": 12,         # Number of layers            (MLP)
    "drop_rate": 0.1,       # Dropout rate                (Regularization)
    "qkv_bias": False       # Query-Key-Value bias        (No bias)
}

In [17]:
import torch 
import torch.nn as nn
from Attention import MultiHeadAttention



class GPTModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    # Word embedding
    self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
    self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

    # Regularizaion
    self.drop_emb = nn.Dropout(cfg["drop_rate"])

    # Transformer
    self.trf_blocks = nn.Sequential(
      *[TransformerBlock(cfg) for I in range(cfg["n_layers"])]
    )

    # Layer normalization 
    self.final_norm = LayerNorm(cfg["emb_dim"])

    # UnEmbedding
    self.out_head = nn.Linear(
      cfg["emb_dim"], cfg["vocab_size"], bias=False
    )

  def forward(self, in_idx):
    batch_size, seq_len = in_idx.shape

    # Process the embeddings 
    tok_embeds = self.tok_emb(in_idx)     # Work embedding 
    # The positional, if the seq_len is smaller than the context_length, we use the seq_len.. 
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
    x = tok_embeds + pos_embeds

    # Regularization
    x = self.drop_emb(x)

    # Transformer blocks 
    x = self.trf_blocks(x)

    # MLP
    x = self.final_norm(x)

    # Logits for the next token prediction
    logits = self.out_head(x)
    return logits




# Transformer block (Multi-head attention and MLP)
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att = MultiHeadAttention(
        d_in=cfg["emb_dim"],
        d_out=cfg["emb_dim"],
        context_length=cfg["context_length"],
        num_heads=cfg["n_heads"], 
        dropout=cfg["drop_rate"],
        qkv_bias=cfg["qkv_bias"]
    )
    self.ff = FeedForward(cfg)
    self.norm1 = LayerNorm(cfg["emb_dim"])
    self.norm2 = LayerNorm(cfg["emb_dim"])
    self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

  def forward(self, x):
    shortcut = x
    x = self.norm1(x)
    x = self.att(x)
    x = self.drop_shortcut(x)
    x = x + shortcut      #2

    shortcut = x         #3
    x = self.norm2(x)
    x = self.ff(x)
    x = self.drop_shortcut(x)
    x = x + shortcut      #4
    return x

  
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module): 
  def __init__(self, cfg):
    super().__init__()
    self.layers = nn.Sequential(
      nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
      GELU(),
      nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
    )

  def forward(self, x):
    return x


# Normal layer - normalize the logits of the final output
class LayerNorm(nn.Module):
  def __init__(self, emb_dim):
    super().__init__()
    self.eps = 1e-5
    self.scale = nn.Parameter(torch.ones(emb_dim))
    self.shift = nn.Parameter(torch.ones(emb_dim))

  def forward(self, x):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    norm_x = (x - mean) / torch.sqrt(var + self.eps)
    return self.scale * norm_x + self.shift


In [9]:
# Test the structure 

import tiktoken


tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

# Encode the text
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))

# Prepare the batch to be process by the GPT
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [ ]:
# Pass the batch to the GPT model using the actual configuration
torch.manual_seed(123)

model = GPTModel(GPT_CONFIG_124M)

# The output of the model ar the logits (scores) for each token on the vocabulary.
logits = model(batch)
print("Output shape:", logits.shape)
print(logits)


Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


In [11]:
# Testing the transformer block
torch.manual_seed(123)
x = torch.rand(2, 4, 768)                   #1
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768])


In [18]:
# Testing the complete GPT Module 
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.8807, -0.7079,  0.4678,  ..., -0.4779, -1.5087,  0.1768],
         [ 0.9326, -1.5775, -0.4821,  ..., -0.5777, -1.2082,  0.2241],
         [ 1.0888, -0.6413, -0.3125,  ...,  0.3281, -2.0400, -0.2065],
         [-0.2673, -0.4456,  0.0524,  ...,  0.7156, -0.8624,  0.2742]],

        [[-0.0845, -1.2975,  0.7583,  ..., -0.1242, -1.3594, -0.2851],
         [ 0.5650, -1.1925, -0.0537,  ...,  0.4740, -1.6781,  0.5950],
         [ 1.2171, -0.4778,  0.0522,  ...,  0.7078, -1.1917,  0.1297],
         [ 0.7208, -1.1088,  0.7357,  ...,  1.2491, -1.9793,  0.3066]]],
       grad_fn=<UnsafeViewBackward0>)


In [20]:
# Paramethers of the current model 
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# The model reuses the weights from the token embedding layer in its output layer

Total number of parameters: 163,009,536


In [21]:
# Memory requirements for the model 
total_size_bytes = total_params * 4       #1
total_size_mb = total_size_bytes / (1024 * 1024)     #2
print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 621.83 MB


## Generating the text - Unembedding 

In [25]:
def generate_text_simple(model, idx, max_new_tokens, context_size): 
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]   
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]                   
        probas = torch.softmax(logits, dim=-1)         
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)   
        idx = torch.cat((idx, idx_next), dim=1)    

    return idx

In [30]:
start_context = "Hello, I am a"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)    #1
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 314, 716, 257]
encoded_tensor.shape: torch.Size([1, 5])


In [31]:
model.eval()                  #1
out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=15, 
    context_size=GPT_CONFIG_124M["context_length"]
)
print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,   314,   716,   257, 18638, 36720, 17260, 29316,  3277,
         25362, 44656, 20089, 20045, 12011, 31263, 29890, 44109, 19343,  7574]])
Output length: 20


In [32]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello, I am a204372 mounting■ nationァ postingshousesMuch MOREulkan Europa flirt lambaning
